# 01 – Data exploration

**Goal:** understand the structure of each raw file before cleaning it.

For every file I check:
- the file format and how the columns are named,
- which column holds the province (needed to join the tables),
- the time period covered and the units,
- anything that could lead to a wrong result if used naively.

Raw files are described in `data/raw/SOURCES.md`. Nothing in `data/raw/` is modified.

In [1]:
import json

import pandas as pd

pd.set_option("display.max_columns", 15)
pd.set_option("display.width", 200)

RAW = "../data/raw/"
TUIK = RAW + "tuik/"

TÜİK bulk-download CSV files are semicolon-separated, and some column names carry a code in brackets, e.g. `Araç Türü (_T)`. A small helper reads a file and keeps only the name part.

In [2]:
def read_tuik_csv(file_name):
    df = pd.read_csv(TUIK + file_name, sep=";", encoding="utf-8-sig", low_memory=False)
    df.columns = [col.split(" (")[0] for col in df.columns]
    return df

## 1. Registered cars by fuel type, Türkiye (YAKIT_CINSI_V4)

Monthly stock of registered cars by fuel type. This is the only TÜİK table with electric cars.

In [3]:
fuel = read_tuik_csv("tuik_DF_MOTORLU_KARA_TASIT_YAKIT_CINSI_V4.csv")
print(fuel.shape)
for col in ["Coğrafi Kapsam", "Yakıt Türü", "Ölçü Birimi"]:
    print(col, ":", fuel[col].unique().tolist())
print("Period:", fuel["Zaman"].min(), "to", fuel["Zaman"].max())

(3332, 14)
Coğrafi Kapsam : ['Türkiye']
Yakıt Türü : ['Toplam', 'Benzinli', 'Dizel', 'LPG', 'Hibrit', 'Elektrik', 'Bilinmeyen']
Ölçü Birimi : ['Yüzde', 'Sayı']
Period: 2005-01 to 2026-09


In [4]:
# Latest month, counts only
latest = fuel[(fuel["Zaman"] == fuel["Zaman"].max()) & (fuel["Ölçü Birimi"] == "Sayı")]
latest[["Zaman", "Yakıt Türü", "Gözlem Değeri"]]

,Zaman,Yakıt Türü,Gözlem Değeri
2278,2026-09,Toplam,18115875
2322,2026-09,Benzinli,5616803
2366,2026-09,Dizel,5793543
2410,2026-09,LPG,5268278
2447,2026-09,Hibrit,915892
2478,2026-09,Elektrik,486359
2516,2026-09,Bilinmeyen,35000


**Observations**
- Coverage is **Türkiye only**. There is no province breakdown of fuel type in any TÜİK table, so electric cars cannot be counted by province.
- Each month appears twice: as a count (`Sayı`) and as a share (`Yüzde`). Only counts will be kept; shares can be recalculated.
- In September 2026 there were **486,359 electric cars out of 18.1 million (2.7%)**. This matches the TÜİK press release.
- The value column is stored as **text**, because the share rows use a decimal comma (`2,68`). Values must be converted to numbers in cleaning.

## 2. All vehicles by fuel type, annual (YAKIT_TURU_V3)

A second fuel table, covering all vehicle types, not only cars.

In [5]:
fuel_all = read_tuik_csv("tuik_DF_MOTORLU_KARA_TASIT_YAKIT_TURU_V3.csv")
print(fuel_all["Zaman"].min(), "to", fuel_all["Zaman"].max())
fuel_all[fuel_all["Zaman"] == 2025][["Yakıt Türü", "Ölçü Birimi", "Gözlem"]]

2005 to 2025


,Yakıt Türü,Ölçü Birimi,Gözlem
0,Toplam,Sayı,33612650
21,Benzinli,Sayı,11854527
42,Dizel,Sayı,14434149
63,LPG,Sayı,5364054
84,Hibrit,Sayı,697318
98,Elektrik,Sayı,1200660
114,Bilinmeyen,Sayı,61942


In [6]:
# Compare electric vehicles (all types) with electric cars at the end of 2025
ev_all_2025 = fuel_all.loc[(fuel_all["Zaman"] == 2025) & (fuel_all["Yakıt Türü"] == "Elektrik")
                           & (fuel_all["Ölçü Birimi"] == "Sayı"), "Gözlem"].iloc[0]
ev_cars_2025 = fuel.loc[(fuel["Zaman"] == "2025-12") & (fuel["Yakıt Türü"] == "Elektrik")
                        & (fuel["Ölçü Birimi"] == "Sayı"), "Gözlem Değeri"].iloc[0]
print("Electric vehicles, all types:", ev_all_2025)
print("Electric cars:              ", ev_cars_2025)

Electric vehicles, all types: 1200660
Electric cars:               370591


**Observations**
- **Data trap:** this table shows about 1.2 million electric vehicles at the end of 2025, but only 370,591 of them are cars. The rest are almost certainly electric motorcycles and scooters, which do not use public car chargers.
- EV counts in this project therefore come **only from the car table** (section 1). This table is not used.

## 3. Registered vehicles by province (ILLER_V3)

Monthly stock of all vehicles by province and vehicle type. This is the main province-level fleet table.

In [7]:
prov = read_tuik_csv("tuik_DF_MOTORLU_KARA_TASIT_ILLER_V3.csv")
print(prov.shape)
for col in ["Araç Türü", "Ölçü Birimi"]:
    print(col, ":", prov[col].unique().tolist())
print("Period:", prov["Zaman"].min(), "to", prov["Zaman"].max())
print("Areas:", prov["Coğrafi Kapsam"].nunique(), "(Türkiye + 81 provinces)")

(214020, 14)
Araç Türü : ['Toplam', 'Otomobil', 'Minibüs', 'Otobüs', 'Kamyonet', 'Kamyon', 'Motosiklet', 'Özel Amaçlı', 'Traktör']
Ölçü Birimi : ['Yüzde', 'Sayı']
Period: 2005-01 to 2026-09
Areas: 82 (Türkiye + 81 provinces)


In [8]:
# Cars by province, latest month: largest five
cars_latest = prov[(prov["Zaman"] == "2026-09") & (prov["Araç Türü"] == "Otomobil") & (prov["Ölçü Birimi"] == "Sayı")].copy()
cars_latest["cars"] = cars_latest["Gözlem Değeri"].astype(int)
print("Sum of 81 provinces:", cars_latest.loc[cars_latest["Coğrafi Kapsam"] != "Türkiye", "cars"].sum())
cars_latest.sort_values("cars", ascending=False)[["Coğrafi Kapsam", "cars"]].head(6)

Sum of 81 provinces: 18115875


,Coğrafi Kapsam,cars
1825,Türkiye,18115875
4435,İstanbul,4210192
61855,Ankara,2271129
20095,İzmir,1130163
69685,Antalya,790152
43585,Bursa,752227


**Observations**
- 81 provinces plus a Türkiye total, monthly from 2005. Province names are in normal case (e.g. `İstanbul`).
- The 81 provinces add up exactly to the Türkiye total (18,115,875 cars), so no vehicles are lost in the province split.
- Istanbul alone has 4.2 million cars (23% of Türkiye). As in the UK project, **company and rental cars are registered at the company's address**, which is often Istanbul. Turkish data has no private/company split, so this cannot be corrected here (it was included in the TÜİK data request).

## 4. First registrations by province (KAYDI_YAPILAN_IL_V2)

Vehicles registered for the first time, by province. Used for growth of the car fleet by province.

In [9]:
reg = read_tuik_csv("tuik_DF_MOTORLU_KARA_TASIT_KAYDI_YAPILAN_IL_V2.csv")
print(reg["Ay"].unique().tolist())

['Ocak', 'Şubat', 'Mart', 'Nisan', 'Mayıs', 'Haziran', 'Temmuz', 'Ağustos', 'Eylül', 'Ekim', 'Kasım', 'Aralık', 'Ocak-Şubat', 'Ocak-Mart', 'Ocak-Nisan', 'Ocak-Mayıs', 'Ocak-Haziran', 'Ocak-Temmuz', 'Ocak-Ağustos', 'Ocak-Eylül', 'Ocak-Ekim', 'Ocak-Kasım', 'Ocak-Aralık']


In [10]:
# Türkiye, cars: the same month appears as a single month and as year-to-date
reg[(reg["Coğrafi Kapsam"] == "Türkiye") & (reg["Araç Türü"] == "Otomobil") & (reg["Ölçü Birimi"] == "Sayı")
    & (reg["Zaman"] == "2026-09")][["Ay", "Zaman", "Gözlem Değeri"]]

,Ay,Zaman,Gözlem Değeri
1825,Eylül,2026-09,64361
4039,Ocak-Eylül,2026-09,665220


**Observations**
- The `Ay` column mixes **single months** (`Eylül`) and **year-to-date totals** (`Ocak-Eylül`) with the same `Zaman` value. Adding them up would double count. Only single months will be kept.
- January–September 2026: 665,220 new cars, which matches the press release.

## 5. EPDK charging station list

Licensed charging stations, read from the EPDK station list on 30 October 2026 (see `SOURCES.md`).

In [11]:
stations = pd.read_csv(RAW + "epdk_sarj_istasyonlari_2026-10-30.csv", encoding="utf-8-sig")
print(stations.shape)
print("Unique station numbers:", stations["istasyon_no"].nunique())
stations.head(3)

(16613, 8)
Unique station numbers: 16613


,istasyon_no,istasyon_adi,hizmet_sekli,marka,sarj_agi_isletmecisi,sarj_istasyonu_isletmecisi,adres,soket_bilgileri
0,ŞRJ/8,TŞOF Sivrihisar DC,Halka Açık,zes,ZES DİJİTAL TİCARET ANONİM ŞİRKETİ,ZES DİJİTAL TİCARET ANONİM ŞİRKETİ,Kurşunlu Mahallesi Ankara-eskişehir Yolu 134. ...,NaN
1,ŞRJ/9,BÜYÜKAĞA PETROL,Halka Açık,ovolt,OVOLT ŞARJ TEKNOLOJİLERİ SANAYİ VE TİCARET ANO...,OVOLT ŞARJ TEKNOLOJİLERİ SANAYİ VE TİCARET ANO...,Camikebir Mahallesi Atatürk Caddesi No:56 Kaym...,NaN
2,ŞRJ/12,"Akbatı AVM, Esenyurt",Halka Açık,eşarj,EŞARJ ELEKTRİKLİ ARAÇLAR ŞARJ SİSTEMLERİ ANONİ...,EŞARJ ELEKTRİKLİ ARAÇLAR ŞARJ SİSTEMLERİ ANONİ...,Sanayi Mahallesi 1655 Sokağı No:6 Esenyurt / İ...,NaN


In [12]:
print(stations["hizmet_sekli"].value_counts())
print("Rows with socket information:", stations["soket_bilgileri"].notna().sum())

hizmet_sekli
Halka Açık    12760
Özel           3853
Name: count, dtype: int64
Rows with socket information: 0


In [13]:
# The province is the last part of the address, after the final "/"
stations["province_raw"] = stations["adres"].str.split("/").str[-1].str.strip()
counts = stations["province_raw"].value_counts()
print("Distinct values:", counts.size)
counts.head(5)

Distinct values: 90


province_raw
İSTANBUL    4433
ANKARA      2115
ANTALYA      916
İZMİR        752
BURSA        699
Name: count, dtype: int64

In [14]:
# Values that are not province names (appear fewer than 3 times)
counts[counts < 3]

province_raw
A ( Ada: - , Pafta: - , Parsel: - )                                                              2
C ( Ada: - , Pafta: - , Parsel: - )                                                              2
1 Sokağı No:21 ( Ada: - , Pafta: - , Parsel: - )                                                 1
Akçay Mahallesi Şehit Pilot Teğmen Tütüncü Caddesi 2 Sokağı ( Ada: - , Pafta: - , Parsel: - )    1
Köyü Karacaahmet Mahallesi M.saip Konukoğlu Sokağı No:33 ( Ada: - , Pafta: - , Parsel: - )       1
1 ( Ada: - , Pafta: - , Parsel: - )                                                              1
Köyü Yeşilöz Mahallesi Cumhuriyet Caddesi No:14 ( Ada: - , Pafta: - , Parsel: - )                1
Köyü Varsaklar Mahallesi Adana Sokağı No:215 ( Ada: - , Pafta: - , Parsel: - )                   1
Köyü Karacaahmet Mahallesi M.saip Konukoğlu Sokağı No:8 ( Ada: - , Pafta: - , Parsel: - )        1
Name: count, dtype: int64

**Observations**
- 16,613 stations, each with a unique station number. **12,760 are public** (`Halka Açık`) and 3,853 private (`Özel`, e.g. company car parks and sites). Only public stations are relevant for public charging.
- **No socket counts:** the socket column is empty because the website shows sockets only in a pop-up for each station. Province comparisons will therefore use **stations**, and sockets and AC/DC only at national level (EPDK report).
- Province names are in upper case (`İSTANBUL`), unlike TÜİK. About 10 addresses do not end with a province name; they need to be fixed by hand in cleaning.
- A station can have several sockets, so stations understate charging points. The national ratio from the EPDK report (sockets ÷ stations) is a useful reference.

## 6. EPDK monthly report (PDF)

The September 2026 report is a PDF of charts. The figures used in the analysis are typed in by hand in notebook 02, with the page number as source:

| Figure | Value (Sep 2026) | Page |
| --- | --- | --- |
| Electric vehicles (all types, TÜİK source) | 491,492 | 2–3 |
| Charging sockets, total / AC / DC | 48,971 / 27,664 / 21,307 | 5 |
| Monthly sockets since Sep 2025 | 35,002 → 48,971 | 5 |
| Installed charging capacity | 3,875 MW | 17 |
| Electricity consumed at chargers | 83,253 MWh | 10 |
| Top 10 provinces by consumption | Istanbul 26.9%, Ankara 13.3%, İzmir 6.1% ... | 16 |

**Observation:** the EPDK electric vehicle count (491,492) is a little higher than TÜİK's electric car count (486,359), probably because it also includes some other vehicle types. The TÜİK car figure is used for the analysis, so that EVs and the total car fleet come from the same table.

## 7. Population, density, urban-rural and household size (ADNKS)

In [15]:
pop = read_tuik_csv("tuik_DF_ADNKS_T35.csv")
print(pop["Yaş Grubu"].unique().tolist(), "| years:", pop["Zaman"].min(), "to", pop["Zaman"].max())
pop[(pop["Zaman"] == 2025) & (pop["Yaş Grubu"] == "Toplam")][["İkamet edilen yer", "Gözlem"]].head(3)

['Toplam', '< 14', '15-64', '65+'] | years: 2007 to 2025


,İkamet edilen yer,Gözlem
18,Toplam,86092168
94,İstanbul,15754053
170,Edirne,422438


In [16]:
density = read_tuik_csv("tuik_DF_ADNKS_T20.csv")
urban = read_tuik_csv("tuik_DF_ADNKS_T37.csv")
hh_size = read_tuik_csv("tuik_DF_ADNKS_T25.csv")
print("Density years:", density["Zaman"].min(), "to", density["Zaman"].max())
print("Urban-rural years:", urban["Zaman"].unique().tolist(), "| categories:", urban["ADNKS Gösterge"].unique().tolist())
print("Household size years:", hh_size["Zaman"].min(), "to", hh_size["Zaman"].max())

Density years: 2007 to 2025
Urban-rural years: [2022, 2023, 2024, 2025] | categories: ['Yoğun kent nüfus', 'Orta kent nüfusu', 'Kır nüfusu']
Household size years: 2008 to 2025


**Observations**
- Population is available to the end of 2025. The total for Türkiye is labelled `Toplam` in this table but `Türkiye` in others.
- Density and household size use a decimal comma (`91,79`) and must be converted.
- The urban-rural table splits each province into **dense urban, medium urban and rural** population (2022–2025). The dense urban share is a simple measure of how urban a province is.

## 8. GDP per capita and Human Development Index (JSON-stat)

These two tables are JSON-stat files: the values are stored in one list, and the dimensions describe how to unfold it into a table. A small helper does this.

In [17]:
def read_jsonstat(file_name):
    with open(TUIK + file_name, encoding="utf-8") as f:
        js = json.load(f)
    dims = js["id"]
    # For each dimension, the category labels in their stored order
    labels = []
    for d in dims:
        cat = js["dimension"][d]["category"]
        index = cat["index"]
        # "index" is either a list of codes in order, or a {code: position} dictionary
        order = index if isinstance(index, list) else sorted(index, key=index.get)
        labels.append([(code, cat["label"][code]) for code in order])
    index = pd.MultiIndex.from_product(labels, names=dims)
    values = pd.Series(js["value"]).astype(float)
    values.index = values.index.astype(int)
    df = pd.DataFrame({"value": values}).reindex(range(len(index)))
    df.index = index
    return df.dropna().reset_index()

In [18]:
gdp = read_jsonstat("tuik_DF_UH_BH_KISI_BASI.json")
print(gdp.shape)
print(gdp["DUZEY_SEVIYE"].apply(lambda x: x[1]).unique().tolist())
gdp_2024 = gdp[(gdp["TIME_PERIOD"].apply(lambda x: x[0]) == "2024")
               & (gdp["DUZEY_SEVIYE"].apply(lambda x: x[1]).str.contains("81"))
               & (gdp["UNIT_MEASURE"].apply(lambda x: x[1]) == "Turkish lira")]
print("Provinces in 2024:", len(gdp_2024))
gdp_2024.head(3)

(6000, 7)
['TR', 'SR-1 (12 Region)', 'SR-2 (26 Region)', 'SR-3 (81 Province)']
Provinces in 2024: 81


,REF_AREA,FREQ,DUZEY_SEVIYE,UNIT_MEASURE,BAZ_YILI,TIME_PERIOD,value
174,"(TR100, İstanbul)","(A, Annual)","(3, SR-3 (81 Province))","(TRY, Turkish lira)","(2009, 2009)","(2024, 2024)",802669.243540
324,"(TR212, Edirne)","(A, Annual)","(3, SR-3 (81 Province))","(TRY, Turkish lira)","(2009, 2009)","(2024, 2024)",417266.190674
374,"(TR213, Kırklareli)","(A, Annual)","(3, SR-3 (81 Province))","(TRY, Turkish lira)","(2009, 2009)","(2024, 2024)",531387.532548


In [19]:
hdi = read_jsonstat("tuik_DF_BESERI_KALKINMA_ENDEKS.json")
print(hdi["INDICATOR"].apply(lambda x: x[1]).unique().tolist())
print("Years:", sorted(hdi["TIME_PERIOD"].apply(lambda x: x[0]).unique()))
hdi.head(3)

['Human Development Index', 'Life Expectancy Index', 'Education Index', 'Income Index']
Years: ['2018', '2019', '2020', '2021', '2022']


,REF_AREA,FREQ,DUZEY_SEVIYE,INDICATOR,ENDEKS_DEGISIM,TIME_PERIOD,value
0,"(TR, Türkiye)","(A, Annual)","(0, TR)","(E_BKE, Human Development Index)","(1, Index)","(2018, 2018)",0.844345
1,"(TR, Türkiye)","(A, Annual)","(0, TR)","(E_BKE, Human Development Index)","(1, Index)","(2019, 2019)",0.845254
2,"(TR, Türkiye)","(A, Annual)","(0, TR)","(E_BKE, Human Development Index)","(1, Index)","(2020, 2020)",0.844218


**Observations**
- Each dimension value is a pair of **(code, label)**. The `REF_AREA` codes are the official SR/NUTS codes (e.g. `TR100` = Istanbul). These codes are a good join key for the 81 provinces.
- GDP per capita is available to **2024**, in Turkish lira and US dollars. Lira values are affected by high inflation, so comparisons between provinces use **one year only** (or ranks).
- The Human Development Index (HDI) goes to **2022** and has three sub-indices (education, income, life expectancy). It plays the role of the English deprivation index (IMD) in the UK project: higher HDI = more developed province.

## 9. Building and Dwelling Characteristics Survey 2021 (BKNA)

Two Excel tables from the survey's press release, by province.

In [20]:
parking = pd.read_excel(TUIK + "tuik_bkna2021_il_otopark.xls", header=None)
parking.head(8)

,0,1,2,3,4,5,6,7
0,İl ve ikamet edilen binada otopark bulunma dur...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Number and proportion of households by provinc...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,İl-Provinces,Konutta ikamet eden hanehalkı sayısı \nNumber ...,Otopark bulunma durumu\nAvailability of car pa...,NaN,NaN,Otopark bulunma durumu (%)\nAvailability of ca...,NaN,NaN
3,NaN,NaN,Otopark var \nCar parking available,Otopark yok\nNo car parking available,NaN,\nToplam\nTotal,Otopark var \nCar parking available,Otopark yok\nNo car parking available
4,Toplam-Total,25329833,10892876,14436957,NaN,100,43.004137,56.995863
5,Adana,632875,331457,301418,NaN,100,52.373247,47.626753
6,Adıyaman,155300,46024,109276,NaN,100,29.635771,70.364229
7,Afyonkarahisar,218348,74793,143555,NaN,100,34.254233,65.745767


In [21]:
floors = pd.read_excel(TUIK + "tuik_bkna2021_il_kat_sayisi.xls", header=None)
floors.iloc[2:7, :10]

,0,1,2,3,4,5,6,7,8,9
2,NaN,Konutta ikamet eden hanehalkı sayısı \nNumber ...,Ortalama \nkat sayısı \nAverage number \nof fl...,Bina kat sayısı-Number of floors in the building,NaN,NaN,NaN,NaN,NaN,NaN
3,İl\nProvinces,NaN,NaN,1,2.0,3.0,4.0,5.0,6+,NaN
4,Toplam-Total,25329833,4.973275,2969100,4391135.0,2873576.0,2718029.0,3658708.0,8719286,NaN
5,Adana,632875,5.430632,85176,223046.0,80613.0,21368.0,12077.0,210594,NaN
6,Adıyaman,155300,3.546189,26904,54767.0,18596.0,10919.0,9164.0,34949,NaN


**Observations**
- **Car park in the building:** 43% of households in Türkiye live in a building with a car park and **57% without one**. This is a much more direct measure of "can this household charge at home?" than the UK project's proxy (flats and terraced houses). It ranges from about 20% without a car park in Ankara to over 70% in some eastern provinces.
- Headers span several rows and are bilingual; data starts on row 5 (index 4). The bottom rows are footnotes.
- Province names have trailing spaces in the floors table (`Adana `).
- The survey year is **2021**, older than the other data, but building stock changes slowly.

## 10. Province names across sources

Each source writes province names differently. This check shows how many names match the TÜİK vehicle table after a simple standardisation (Turkish upper case, spaces removed).

In [22]:
def standard_name(name):
    # Turkish-aware upper case: i -> İ and ı -> I must be handled before .upper()
    name = str(name).strip().replace("i", "İ").replace("ı", "I")
    return name.upper()

reference = set(prov.loc[prov["Coğrafi Kapsam"] != "Türkiye", "Coğrafi Kapsam"].map(standard_name))
print("Provinces in vehicle table:", len(reference))

sources = {
    "EPDK stations": stations["province_raw"],
    "Population": pop["İkamet edilen yer"],
    "Parking (BKNA)": parking.iloc[5:, 0],
    "GDP (JSON-stat)": gdp_2024["REF_AREA"].apply(lambda x: x[1]),
}
for source, names in sources.items():
    names = set(names.dropna().map(standard_name))
    print(f"{source:16s} matched: {len(reference & names):2d} / 81 | unmatched examples: {sorted(reference - names)[:4]}")

Provinces in vehicle table: 81
EPDK stations    matched: 81 / 81 | unmatched examples: []
Population       matched: 81 / 81 | unmatched examples: []
Parking (BKNA)   matched: 81 / 81 | unmatched examples: []
GDP (JSON-stat)  matched: 81 / 81 | unmatched examples: []


**Observations**
- After this simple standardisation, **all four sources match all 81 provinces**. Notebook 02 will add the official SR code (e.g. `TR100`) from the JSON-stat files to every table, so all tables join on a code rather than a name. Only the ~10 broken EPDK addresses (section 5) need fixing by hand.

## Summary

| Source | Level | Latest period | Key points for cleaning |
| --- | --- | --- | --- |
| Cars by fuel type (TÜİK) | Türkiye | Sep 2026 | Only national EV data. Keep counts, drop shares |
| All vehicles by fuel (TÜİK) | Türkiye | 2025 | Not used: "electric" includes motorcycles |
| Vehicles by province (TÜİK) | 81 provinces | Sep 2026 | Cars by province; company cars registered at head office |
| First registrations by province (TÜİK) | 81 provinces | Sep 2026 | Drop year-to-date rows to avoid double counting |
| Charging stations (EPDK) | Station / province | 30 Oct 2026 | 12,760 public; no sockets; province from address; ~10 addresses to fix |
| Monthly charging report (EPDK) | Türkiye | Sep 2026 | Sockets AC/DC, capacity; typed in by hand |
| Population, density, urban-rural, household size (TÜİK) | 81 provinces | 2025 | Decimal commas; `Toplam` = Türkiye |
| GDP per capita (TÜİK) | 81 provinces | 2024 | JSON-stat; has SR codes; use one year |
| Human Development Index (TÜİK) | 81 provinces | 2022 | JSON-stat; replaces IMD |
| Car park and floors (TÜİK BKNA) | 81 provinces | 2021 | Multi-row headers; share without car park = home charging proxy |

**Main limitation:** there is no electric car count by province. Province-level questions use public charging stations per 10,000 cars and per 100,000 people, and the national data shows how fast EVs are growing compared with sockets.